# A/B Testing Activation Calculation From Local CSV Backup

This backup version reads `users_raw.csv` and `tracks_raw.csv` from local files, then calculates activation in pandas.


In [ ]:
import hashlib
from pathlib import Path

import pandas as pd

DATA_DIR = Path("lightdash-demo-saas/seeds")

users = pd.read_csv(DATA_DIR / "users_raw.csv", parse_dates=["first_logged_in_at"])
tracks = pd.read_csv(DATA_DIR / "tracks_raw.csv", parse_dates=["event_timestamp"])

def assign_variant(user_id: str) -> str:
    hash_value = int(hashlib.md5(user_id.encode()).hexdigest(), 16)
    return "treatment" if hash_value % 2 else "control"

assignments = users[["user_id"]].copy()
assignments["variant"] = assignments["user_id"].apply(assign_variant)

activation_events = tracks[tracks["event_name"].isin(["workspace_created", "report_generated"])]
activation_events = activation_events.merge(
    users[["user_id", "first_logged_in_at"]],
    on="user_id",
    how="inner",
)

activation_events = activation_events[
    (activation_events["event_timestamp"] >= activation_events["first_logged_in_at"])
    & (activation_events["event_timestamp"] <= activation_events["first_logged_in_at"] + pd.Timedelta(days=7))
]

user_activation = (
    activation_events
    .pivot_table(
        index="user_id",
        columns="event_name",
        values="event_timestamp",
        aggfunc="min",
    )
    .reset_index()
)

user_activation["is_activated"] = (
    user_activation["workspace_created"].notna()
    & user_activation["report_generated"].notna()
)

experiment_users = users[["user_id", "first_logged_in_at"]].merge(assignments, on="user_id")
experiment_users = experiment_users.merge(
    user_activation[["user_id", "is_activated"]],
    on="user_id",
    how="left",
)
experiment_users["is_activated"] = experiment_users["is_activated"].fillna(False).astype(bool)

summary = (
    experiment_users
    .groupby("variant", as_index=False)
    .agg(
        users=("user_id", "count"),
        activated_users=("is_activated", "sum"),
    )
)
summary["activation_rate"] = summary["activated_users"] / summary["users"]
summary
